In [6]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


C:\Users\Dell\AppData\Local\Temp\ipykernel_16624\2389807203.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [7]:
load_dotenv()

True

In [8]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environment variable loaded :)")

Environment variable loaded :)


In [10]:
#Loading our data file
DATA_DIR = os.path.join(os.getcwd(), "data","hr_policy.txt")

In [11]:
#data ingestion
loader = TextLoader(DATA_DIR,encoding="utf-8")
documents = loader.load()

print(documents)


[Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from th

In [12]:
print(documents[0].metadata)

{'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require writte

In [14]:
len(chunks)
print(chunks[1])

page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}


In [15]:
##Embed our data using Jina Embeddings
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("embeddings model loaded :)",embeddings_model.model_name)


embeddings model loaded :) jina-embeddings-v2-base-en


In [16]:
#Store data in vector database using Groq
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings_model)
print("vector store created :)",vector_store.index.ntotal)

vector store created :) 9


In [17]:
test_query = "How many sick leaves employees get?"
#similarity search
similar_docs = vector_store.similarity_search(test_query, k=3)
print("similar docs :)",similar_docs)
for i,match in enumerate(similar_docs):
    print(f"Match {i+1}:")
    print(f"Content: {match.page_content}")
    print(f"Metadata: {match.metadata}")
    print() 

similar docs :) [Document(id='d7e7b0eb-ce4a-43bd-9749-e87942e30e5a', metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(id='91564c64-1215-488f-b996-c4bc3c0d4603', metadata={'source': 'd:\\langchian project\\RagHRPolicy\\data\\hr_policy.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.'), Docume

In [ ]:
#TOOL
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [18]:
#Data Retrieval using RAG
#LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.9,
)

llm.model_name

'openai/gpt-oss-120b'

In [19]:
test_response = llm.invoke("hey is learning Rag hard? answer in one line")
test_response.content

'Learning RAG can be challenging at first, but with focused practice it becomes manageable.'

In [20]:
#AI AGENT 3 LLM - brain , tool - super power , memory - no memory
from langchain.agents import create_agent


In [ ]:
hr_assistant = create_agent(
    model=llm,
    tools = [search_hr_policy],
    system_prompt = """ You are a helpful HR assistant. You have access to the company's HR policy documents.
       Use the information from these documents to answer any questions related to HR policies, 
       procedures, and guidelines. If you don't know the answer, respond with "I don't know."   
    """,
)
print("HR assistant is ready to answer your questions :)")
    

HR assistant is ready to answer your questions :)


In [22]:
def ask_hr_assistant(question:str) -> str:
    """send a question to the RAG and print the answer"""
    print("="*60)
    print("Question:",question)
    print("-" * 60)
    
    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content
    
    print("Answer:",answer)
    print("="*60)
    
    print()
    return answer

In [ ]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)
#answer = response

In [26]:
response

{'messages': [HumanMessage(content='How are you', additional_kwargs={}, response_metadata={}, id='56ebd45b-5664-45f0-a485-dc8f719b4c26'),
  AIMessage(content='I’m doing well, thank you! How can I assist you with HR‑related questions today?', additional_kwargs={'reasoning_content': 'The user asks "How are you". This is a casual greeting, not a HR policy question. The developer instruction says: "You are a helpful HR assistant. You have access to the company\'s HR policy documents. Use the information from these documents to answer any questions related to HR policies, procedures, and guidelines. If you don\'t know the answer, respond with "I don\'t know.""\n\nBut the user is just asking a casual question. The system prompt says we should be friendly. The developer instruction does not forbid answering small talk. It says you are a helpful HR assistant, but can answer any questions related to HR policies; if you don\'t know, respond with "I don\'t know". For non-HR questions, maybe we ca

In [27]:
#SYSTEM MESSAGE - HR ASSISNT
#HUMAN MESSAGE - TELL ME ABOUT POLICIES
#AI MESSAGE - HEY THESE ARE THEPLOICES
response["messages"][-2].content

'How are you'